Below are the user defined parameters we need to be able to change at the hotfire to tune the ereg. These are subject to change and hence shouldn't be hardcoded. I've assumed that the pressures are read in Pascals. I think the PT gives output in bar, if it does, switch to SI units. All the formulae implemented are in SI units.

In [ ]:
# Command Timings
mainValveTime = 2 #s #When the main valves are opened, you probably know how to implement this better, eregValveDelay below is wrt to this. It represents when the ereg controller
#gets authority over the servo, before this servos will be commanded shut.
eregValveDelay = -40e-03 #s Wil probably be zero (i.e. at same time as fuel and ox main valves, but would be good to keep this so we can play around with it at Silwood)

# Gains
Kp1 = 1
Kp2 = 10
KpRampTime = 3 #s
Kc = 8

#Gain Scheduling has been implemented, linearly change from Kp1 to Kp2 over KpRampTime,
#if making it linear is difficult, change it over steps (5-10 steps in between shouldn't change controller). The ramping should start as soon as ereg is given authority, i.e. mainValveTime + eregValveDelay
#Kp is for the Proportional section of the controller defined below.
#Kc is CdA Gain/Correction factor be applied before using the LUT


CdAArray = [0, 0, 0, 4.0e-6, 12.7e-6, 15.2e-6, 17.1e-6, 18.6e-6, 19.5e-6, 19.8e-6]; #Array to be implemented, SI units, equivalent to CdA*Kc
ThetaArray = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]; #Valve Angle in degrees, corresponding to CdA*Kc #Will provide these later


# Pressure Settings
# Nitrogen
nitrogenPressure = 300; #bar
nitrogenPressure = nitrogenPressure * 1e+05; #Pa

#Fuel
fuelSetpoint = 60; #bar
fuelSetpoint = fuelSetpoint * 1e+05; #Pa
fuelRho = 790; #kg/m^3
fuelMdot = 0.79; #kg/s

#Ox
oxSetpoint = 45; #bar
oxSetpoint = oxSetpoint * 1e+05; #Pa
oxRho = 963; #kg/m^3
oxMdot = 2.35; #kg/s

#HardStops
eregHardStop = 45 #Maximum commanded angle we want the controller to be able to give so we dont blow the burst disk


: 

Assuming the following variables are measurements from Pressure Transducers, variables initialised with approx values so the jupyter notebook can run if you need it to.

In [ ]:
nitrogenTankReading = 300 * 1e+05 #Pa
oxTankReading = 40 * 1e+05 #Pa
fuelTankReading = 40*1e+05 #Pa

Okay below's the actual code that the ereg implements, this is for fuel, and for ox just switch out with ox constants and ox tank readings. I've added in code from my Matlab sim implementation where possible so you can refer to it if needed. 

In [ ]:
#Getting Feed Forward Angle first
import numpy as np

CdA = (fuelMdot / fuelRho) * 1/np.sqrt(max(0,nitrogenTankReading)) * 0.673
CdA_corrected = CdA * Kc

def LUT(CdA_corrected):
    pass  #Implement the linear interpolation look up table here, where it takes in the corrected CdA and gives out the Angle in degrees based on constant arrays provided above

ThetaFF = LUT(CdA_corrected) #Get the feed forward angle command

======================MATLAB CODE==========================================
THIS IS HOW THE LUT IS IMPLEMENTED IN MY MATLAB CODE, SOMETHING SIMILAR ON THE BOARD, YOU KNOW BETTER

function [valveAngle, CdA] = feedForwardAngle(n2Pressure, CdACorrectionFactor, CdAArray, mdotProp, rhoProp)
    % Required effective flow area
    CdA = (mdotProp / rhoProp) * 1/sqrt(max(0,n2Pressure)) * 0.673;
    CdA = CdA*CdACorrectionFactor;
% Ball valve opening angle [deg]
    thetaArray = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90];

    theta = 0;

    % Clamp
    if CdA <= CdAArray(1)

        theta = thetaArray(1);

    elseif CdA >= CdAArray(end)

        theta = thetaArray(end);

    else

        % Linear interpolation between LUT points
        for i = 1:length(CdAArray)-1

            if CdA >= CdAArray(i) && CdA <= CdAArray(i+1)

                theta = thetaArray(i) + ...
                    ((CdA - CdAArray(i)) / ...
                    (CdAArray(i+1) - CdAArray(i))) * ...
                    (thetaArray(i+1) - thetaArray(i));

                break;

            end

        end

    end

    % Output angle in radians
    valveAngle = deg2rad(theta);

end

================================================================================

In [ ]:
#Cool, now getting feedback angle
nonDimError = (1 - fuelTankReading/fuelSetpoint) #Non dimensionalised error so we can tune the controller easier
ThetaFB = Kp * nonDimError * eregHardStop #Here Kp should be whatever the gain scheduling commands it to be at the time based on what was said above, multiplied by eregHardStop because if error is 1, we want it to open fully

#Now, both our angles are in degrees
#Total angle command
ThetaTotal = ThetaFB + ThetaFF

#THIS IS WHAT THE EREG COMMANDS IT TO BE